In [ ]:
using LinearAlgebra
using PythonPlot

In [ ]:
g = 9.8
Ac = [0 1; 0 0]
cc = [0; -g]

In [ ]:
#Create discrete-time dynamics model
Δt = 0.1
M = exp([Ac cc; 0 0 0]*Δt)
A = M[1:2,1:2]
c = M[1:2,3]

In [ ]:
#Noise covariances
V = Diagonal([0.01; 0.01]) 
W = 1.0

In [ ]:
#Initial conditions
x0 = randn(2)

In [ ]:
#Simulate
Tfinal = 2.0
Nt = 21
xtraj = zeros(2,Nt)
xtraj[:,1] = x0
for k = 1:(Nt-1)
    xtraj[:,k+1] .= A*xtraj[:,k] + c + sqrt(V)*randn(2)
end

In [ ]:
times = LinRange(0,Tfinal,Nt)
plot(times, xtraj[1,:], label='q')
plot(times, xtraj[2,:], label='v')
legend()

In [ ]:
#Generate noisy measurements
ytraj = zeros(Nt)
for k = 1:Nt
    ytraj[k] = xtraj[1,k] + sqrt(W)*randn()
end

In [ ]:
plot(times, xtraj[1,:], label='q')
plot(times, ytraj, label='y')
legend()

In [ ]:
#Define the measurement model
C = [1.0 0]

In [ ]:
#Filter Initialization
x̄ = zeros(2,Nt)
x̄[:,1] = xtraj[:,1] + randn(2)

P = zeros(2,2,Nt)
P[:,:,1] .= 1.0*I(2)

In [ ]:
#Run filter on data
for k = 1:(Nt-1)
    #Prediction
    xpred = A*x̄[:,k] + c
    Ppred = A*P[:,:,k]*A' + V

    #Innovation
    z = ytraj[k+1] - (C*xpred)[1]
    S = (C*Ppred*C')[1] + W

    #Kalman gain
    L = Ppred*C'/S

    #Update
    x̄[:,k+1] .= xpred + L*z
    P[:,:,k+1] .= (I-L*C)*Ppred*(I-L*C)' + L*W*L'
end

In [ ]:
# Plot the estimate vs. the ground truth
plot(times, xtraj[1,:], label="qtrue")
plot(times, x̄[1,:], label="qfilt")
plot(times, ytraj, label='y')
legend()

In [ ]:
plot(times, xtraj[2,:], label="vtrue")
plot(times, x̄[2,:], label="vfilt")
legend()

In [ ]:
#Plot covariance components
plot(times, P[1,1,:], label="P11")
plot(times, P[2,2,:], label="P22")
legend()